# 🚀 Rocket Engine Safety Shutdown (Reflex Agent)
LangGraph + Groq + Llama 3.1 8B Instant

Designed as a Reflex Agent for mission-critical safety.

In [ ]:
!pip install langgraph langchain langchain-core langchain-groq gradio

In [ ]:
import os
from typing import TypedDict

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, END

import gradio as gr

In [ ]:
# Set your Groq API key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [ ]:
class RocketState(TypedDict):
    temperature: float
    threshold: float
    status: str
    decision: str
    llm_output: str

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)

In [ ]:
def safety_check(state: RocketState):
    temp = state["temperature"]
    threshold = state["threshold"]

    if temp > threshold:
        state["decision"] = "SHUTDOWN"
        state["status"] = "CRITICAL"
    else:
        state["decision"] = "CONTINUE"
        state["status"] = "NORMAL"

    return state

In [ ]:
def generate_llm_output(state: RocketState):
    prompt = f"""
    You are an aerospace safety AI system.

    Analyze the rocket engine condition:

    - Current Temperature: {state['temperature']} °C
    - Safety Threshold: {state['threshold']} °C
    - Status: {state['status']}
    - Decision: {state['decision']}

    Provide a professional aerospace-grade response including:
    - Risk level
    - Explanation
    - Recommended action
    - Tone: Mission Control System
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    state["llm_output"] = response.content

    return state

In [ ]:
builder = StateGraph(RocketState)

builder.add_node("safety_check", safety_check)
builder.add_node("llm_output", generate_llm_output)

builder.set_entry_point("safety_check")

builder.add_edge("safety_check", "llm_output")
builder.add_edge("llm_output", END)

graph = builder.compile()

In [ ]:
def run_system(temp, threshold):
    result = graph.invoke({
        "temperature": temp,
        "threshold": threshold
    })

    return result["llm_output"]

In [ ]:
def launch_ui():
    with gr.Blocks(theme=gr.themes.Soft()) as demo:

        gr.HTML("""
        <div style="text-align:center; padding:20px;">
            <h1 style="color:#0B3D91;">🚀 Rocket Engine Safety Shutdown</h1>
            <h3 style="color:gray;">Collins Aerospace AI Control System</h3>
            <marquee style="color:#0B3D91; font-weight:bold;">
                --- Real-Time Engine Monitoring | Autonomous Safety Reflex System ---
            </marquee>
        </div>
        """)

        with gr.Row():
            temp_input = gr.Slider(0, 2000, value=500, label="Engine Temperature (°C)")
            threshold_input = gr.Slider(0, 2000, value=800, label="Safety Threshold (°C)")

        output_box = gr.Textbox(label="AI Safety Report", lines=15)

        run_btn = gr.Button("Run Safety Check")

        run_btn.click(
            fn=run_system,
            inputs=[temp_input, threshold_input],
            outputs=output_box
        )

    return demo

app = launch_ui()
app.launch()